# Episode 6 — `lores~`: the Filter and the Sample Loop

⏱ About 30 minutes · Difficulty ★★★

**Musical element this episode controls: brightness and resonance (the "nasal" ring).**

**The equivalent Max patch:** saw → `onepole~`, then saw → `lores~` (or `svf~`).

**What you will hear:**
1. the raw bright saw, and the same note made dull by a lowpass
2. with resonance, a "nasal / whistling" peak near the cutoff
3. white noise turned into "wind" and a "whistle" by the filter

**By the end you can:**
- explain why a filter needs a loop while an oscillator does not
- read a one-pole lowpass line by line
- match it to `history` in gen~
- use a `lowpass()` with resonance
- see what a filter did by looking at a spectrum

This is the episode with the most new concepts in the series. Take your time.

## Toolbox

Everything built in earlier episodes, copied as-is. Just run it; no need to re-read (the tricky lines carry `#` comments).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip holds anything beyond ±1 so nothing blows up
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # only plot this many samples from the start
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

import time

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # phase of every table position, 0 to 1
    table = np.zeros(N)                        # start from an empty table of zeros
    for k, amp in enumerate(harmonic_amps, start=1):
        # harmonic k: k cycles across the table, at level amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize: peak becomes 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # how many harmonics frame f gets: rising evenly from 1 to max_harmonics
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # stack into a 2-D array: F rows × N columns

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- which two frames are we between ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # turn 0..1 into a frame number 0..F-1, decimals allowed
    f0 = np.floor(fpos).astype(int)            # the frame below
    f1 = np.minimum(f0 + 1, F - 1)             # the frame above (never past the last one)
    ffrac = fpos - f0                          # how far above the lower frame, 0..1

    # --- which two positions inside a table (same as Episode 3) ---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # left position
    i1 = (i0 + 1) % N                          # right position, wrapping to 0 at the end
    frac = p - np.floor(p)                     # the fractional part

    # interpolate inside each frame first, then between the two frames
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

frames = make_frames()

freq = 110
dur = 2.0
t = np.arange(int(sr * dur)) / sr
phase = (freq * t) % 1

saw = osc_wavetable(frames, phase, 1.0)

`saw` is the last wavetable frame (32 harmonics), 2 seconds, 110 Hz. It is the filter input throughout this episode.

## 1. Why a filter is different

So far every sample could be computed **independently**: `x[n]` depended only on `t[n]`, not on any other sample. That is why whole arrays could be computed at once with no loop.

A filter cannot work that way. Its output depends on **its own previous output** — this is feedback.

To compute output 100 you need output 99; to compute 99 you need 98 ... So the only way is to **start at the beginning and go one by one, in order**.

In Max the audio engine already works like this and you never notice. You met it when using `history` inside `gen~`: `history` is "remember the result of the previous sample".

In Python, "one by one, in order" is a `for` loop.

## 2. The `for` loop

`for n in range(4):` means: let `n` be 0, 1, 2, 3 in turn, running the indented lines each time.

In [ ]:
for n in range(4):
    print("this is round", n)

A variable defined **outside** the loop and changed inside it is **carried into the next round**. That is how you "remember the last time":

In [ ]:
total = 0                          # outside the loop: runs once
for n in range(4):
    total = total + 10             # each round adds 10 to the previous round's result
    print("round", n, "  total is now", total)

`total` is a `history`.

## 3. The simplest filter: a one-pole lowpass

For each sample it does one thing:

> **new output = previous output + a × (input − previous output)**

In words: look how far the input is from where I am now, and **take a small step toward it**. `a` sets the size of the step (0 to 1).

### Walk through it with 10 numbers

The input is a "step": two zeros, then a sudden jump to 1. `a = 0.5`, so each step covers half the remaining distance.

In [ ]:
xs = [0, 0, 1, 1, 1, 1, 1, 1, 1, 1]
a = 0.5

prev = 0.0                                     # the previous output; starts at 0
for n in range(len(xs)):
    distance = xs[n] - prev                    # how far the input is from where I am
    prev = prev + a * distance                 # step toward it by a fraction a
    print(f"n = {n}   input = {xs[n]}   output = {prev:.3f}")

Read the output: the input jumps to 1 **instantly** at `n = 2`. The output does not jump with it; it goes 0.5 → 0.75 → 0.875 → ... creeping toward 1.

**An "instant jump" has been worn into a "smooth rise".** A jump means high frequencies. So this thing wears away highs — it is a lowpass.

### The size of `a`

The smaller `a`, the smaller the step, the worse it follows fast changes, and the more highs it removes.

In [ ]:
step = np.concatenate([np.zeros(20), np.ones(180)])        # 20 zeros, then 180 ones

plt.figure(figsize=(9, 3))
plt.plot(step, color="gray", label="input")
for a in [0.5, 0.1, 0.02]:
    out = np.zeros(len(step))
    prev = 0.0
    for n in range(len(step)):
        prev = prev + a * (step[n] - prev)
        out[n] = prev                                      # store this round's result in slot n of the output array
    plt.plot(out, label=f"a = {a}")
plt.legend()
plt.xlabel("sample")
plt.show()

### Wrap it in a function: `onepole~`

The user wants to give a cutoff in Hz, not `a`. The first line converts cutoff to `a`: the higher the cutoff, the closer `a` is to 1 (bigger steps, more highs let through). No need to memorise the formula.

In [ ]:
def onepole(x, cutoff):
    a = 1 - np.exp(-2 * np.pi * cutoff / sr)       # cutoff (Hz) → step size a (0..1)
    y = np.zeros(len(x))                           # prepare an empty output array
    prev = 0.0                                     # = history in gen~
    for n in range(len(x)):                        # n = 0, 1, 2 ... 88199
        prev = prev + a * (x[n] - prev)            # one small step toward the input
        y[n] = prev                                # store this sample's output
    return y

for cutoff in [100, 1000, 10000]:
    print(f"cutoff {cutoff:>5} Hz  ->  a = {1 - np.exp(-2 * np.pi * cutoff / sr):.3f}")

### Side by side with gen~

The same filter in a gen~ codebox looks like this:

```
History prev(0);
a = 1 - exp(-twopi * in2 / samplerate);
y = prev + a * (in1 - prev);
prev = y;
out1 = y;
```

| gen~ | Python |
|---|---|
| `History prev(0);` | `prev = 0.0` outside the loop |
| this code runs once per sample automatically | you write `for n in range(len(x)):` yourself |
| `in1` | `x[n]` |
| `out1 = y;` | `y[n] = prev` |

**The only difference: gen~'s loop is hidden; in Python you write it out.**

### Listen

**You should hear:** the raw bright saw first; then (cutoff 300 Hz) the same note, clearly duller and more distant.

In [ ]:
display(play(saw))
display(play(onepole(saw, 300)))

**You should see:** the saw's sharp corners rounded off.

In [ ]:
n = int(sr * 0.03)
plt.figure(figsize=(9, 3))
plt.plot(t[:n], saw[:n], label="saw")
plt.plot(t[:n], onepole(saw, 300)[:n], label="onepole 300 Hz")
plt.legend()
plt.xlabel("time (s)")
plt.show()

### The price of a loop

Whole-array math is fast; loops are slow. Measure it:

In [ ]:
start = time.time()
_ = np.sin(2 * np.pi * 110 * t)                # no feedback: the whole array at once
print(f"whole-array sine : {1000 * (time.time() - start):.2f} ms")

start = time.time()
_ = onepole(saw, 300)                          # feedback: 88200 loop rounds
print(f"one-pole loop    : {1000 * (time.time() - start):.2f} ms")

The loop is tens to hundreds of times slower. For a 2-second sound it is still the blink of an eye, but this is why you **avoid loops whenever you can** and write them only where feedback forces you to.

## 4. Highpass: a free extra

A lowpass keeps the lows. **The original minus the lowpassed signal leaves the highs** — a highpass.

**You should hear:** a thin, fizzy sound with the low "body" gone.

In [ ]:
highpassed = saw - onepole(saw, 2000)
play(highpassed)

## 5. Seeing what a filter did: the spectrum

In Max you would patch in a `spectroscope~`. Here is a matching function: a sound goes in, "how loud is each frequency" comes out.

The FFT inside is not explained here; **treat it as a ready-made object for now**. Episode 10 says a little more.

- horizontal: frequency (Hz)
- vertical: level (dB); 0 is the loudest frequency, lower means weaker

**You should see:** the saw as a row of evenly spaced spikes (harmonics) getting slowly shorter to the right. After the one-pole, spikes above 300 Hz are pushed down, more so the further right you go.

In [ ]:
def spectrum(x):
    X = np.abs(np.fft.rfft(x * np.hanning(len(x))))        # the strength of every frequency
    f = np.fft.rfftfreq(len(x), 1 / sr)                    # the matching frequency axis (Hz)
    return f, 20 * np.log10(X / np.max(X) + 1e-9)          # converted to dB, loudest = 0

def spec_plot(signals, max_freq=5000):
    plt.figure(figsize=(9, 4))
    for name in signals:
        f, db = spectrum(signals[name])
        plt.plot(f, db, label=name, linewidth=0.8)
    plt.xlim(0, max_freq)
    plt.ylim(-90, 5)
    plt.xlabel("frequency (Hz)")
    plt.ylabel("level (dB)")
    plt.legend()
    plt.show()

spec_plot({"raw saw": saw, "onepole 300 Hz": onepole(saw, 300)})

## 6. A filter with resonance: `lores~` / `svf~`

The one-pole is gentle (only 6 dB per octave) and has no resonance. Synths normally use a 2-pole **state variable filter** (`svf~`): steeper (12 dB per octave) and able to push up a resonant peak at the cutoff.

Its structure is **exactly the same** as the one-pole: a loop that remembers the previous state. The only differences:

- two state values instead of one (`ic1`, `ic2`)
- five lines of arithmetic per step instead of one

You would not open `lores~` in Max to look inside. Same here: **knowing it is "a loop plus two history values" is enough**; the arithmetic itself need not be understood. Every part of the function is commented.

- `cutoff`: Hz (one number, or an array — next episode uses that)
- `res`: 0 to 1; the higher, the sharper the resonant peak

In [ ]:
def lowpass(x, cutoff, res=0.0):
    x = np.asarray(x, dtype=float)
    n_samples = len(x)
    # cutoff may be one number or an array: make it an array as long as the sound
    cutoff = np.clip(np.broadcast_to(cutoff, n_samples), 20, sr * 0.45)

    # --- turn cutoff / res into the filter's internal coefficients (one set per sample) ---
    g = np.tan(np.pi * cutoff / sr)
    k = 1.414 * (1 - res) + 0.05 * res         # damping: higher res → smaller k → stronger resonance
    a1 = 1 / (1 + g * (g + k))
    a2 = g * a1
    a3 = g * a2

    # .tolist() turns arrays into plain lists: much faster to read one by one in a loop
    xs, A1, A2, A3 = x.tolist(), a1.tolist(), a2.tolist(), a3.tolist()
    y = [0.0] * n_samples
    ic1 = ic2 = 0.0                            # two state variables = two history objects in gen~

    for n in range(n_samples):                 # one sample at a time, in order
        v3 = xs[n] - ic2
        v1 = A1[n] * ic1 + A2[n] * v3
        v2 = ic2 + A2[n] * ic1 + A3[n] * v3    # v2 is the lowpass output
        ic1 = 2 * v1 - ic1                     # update the state for the next sample
        ic2 = 2 * v2 - ic2
        y[n] = v2
    return np.array(y)

### Listen: cutoff

**You should hear:** three notes getting brighter — 200 Hz is very dull, 800 Hz medium, 4000 Hz close to the raw saw.

In [ ]:
for cutoff in [200, 800, 4000]:
    print("cutoff", cutoff, "Hz")
    display(play(lowpass(saw, cutoff)))

### Listen: resonance

Cutoff fixed at 800 Hz, only `res` changes.

**You should hear:** `res = 0` is a plain dull tone. As `res` rises, an increasingly clear "nasal / whistling" peak appears near 800 Hz, like a mouth shaping an "eh".

(High resonance raises the peak level, so the gain is lowered for the last two.)

In [ ]:
display(play(lowpass(saw, 800, res=0.0)))
display(play(lowpass(saw, 800, res=0.6), gain=0.2))
display(play(lowpass(saw, 800, res=0.9), gain=0.15))

**You should see:** the three curves are similar below 800 Hz; the higher `res`, the taller the peak pushed up at 800 Hz.

In [ ]:
spec_plot({
    "res 0.0": lowpass(saw, 800, 0.0),
    "res 0.6": lowpass(saw, 800, 0.6),
    "res 0.9": lowpass(saw, 800, 0.9),
}, max_freq=3000)

## 7. Filtering noise

`rng.uniform(-1, 1, n)`: `n` random numbers between -1 and 1. Each sample is unrelated to its neighbours — that is white noise, `noise~` in Max. It contains all frequencies.

**You should hear:**
1. raw white noise: "shhh"
2. lowpass at 400 Hz: a dull rumble, like distant wind or surf
3. lowpass at 1200 Hz with very high resonance: a pitched "whistle" (near 1200 Hz) rising out of the noise

In [ ]:
rng = np.random.default_rng(0)
noise = rng.uniform(-1, 1, len(t))

display(play(noise, gain=0.15))
display(play(lowpass(noise, 400), gain=0.5))
display(play(lowpass(noise, 1200, res=0.98), gain=0.15))

## 8. Two filters in series

Send a filter's output into another identical filter and the slope doubles (24 dB per octave). In Max: two `lores~` in a row.

**You should hear:** the second one duller and "rounder" than the first.

In [ ]:
once = lowpass(saw, 600)
twice = lowpass(once, 600)

display(play(once))
display(play(twice))

spec_plot({"raw saw": saw, "one filter": once, "two filters": twice}, max_freq=4000)

## Common mistakes

| Symptom | Cause |
|---|---|
| The filter output is all zeros | `y[n] = prev` is missing, or sits outside the loop (wrong indentation) |
| The output is only quieter, not duller | `prev = 0.0` sits **inside** the loop and is reset every round, so nothing is "remembered" |
| It runs very slowly | Unnecessary numpy work inside the loop, or a very long sound |
| Distortion at high resonance | The peak exceeds 1; lower the gain |

Indentation has meaning in Python: indented lines belong to the loop; unindented lines run after the loop has finished.

## Exercises

**1.** Set the `onepole` cutoff to 50, 500 and 5000 and listen. **Guess first:** which is closest to the raw saw?

<details><summary>Answer</summary>

```python
for c in [50, 500, 5000]:
    display(play(onepole(saw, c)))
```
5000 is closest. The higher the cutoff, the more gets through.
</details>

**2.** Build a resonant highpass with `saw - lowpass(saw, 1500, res=0.5)`; listen and plot its spectrum.

<details><summary>Answer</summary>

```python
hp = saw - lowpass(saw, 1500, res=0.5)
play(hp, gain=0.2)
spec_plot({"highpass": hp})
```
</details>

**3.** Change the cutoff of the "whistle" in section 7 to 300 and 3000. **You should hear:** the whistle's pitch following the cutoff.

<details><summary>Answer</summary>

```python
for c in [300, 1200, 3000]:
    display(play(lowpass(noise, c, res=0.98), gain=0.15))
```
</details>

## Recap

| Max | Python |
|---|---|
| the audio engine running per sample | `for n in range(len(x)):` |
| `history` (gen~) | a variable defined outside the loop and updated inside it |
| `onepole~` | `onepole(x, cutoff)` |
| `lores~` / `svf~` | `lowpass(x, cutoff, res)` |
| `noise~` | `rng.uniform(-1, 1, n)` |
| `spectroscope~` | `spec_plot({...})` |

**The rule:** no feedback → compute whole arrays at once. Feedback → you need a loop.

**Next episode:** patching envelopes and LFOs into cutoff and pitch — modulation.